In [152]:
import pandas as pd

rawsearch_engine_map = pd.read_csv("../resources/wum_dataset_hw/search_engine_map.csv")
rawvisitors = pd.read_csv("../resources/wum_dataset_hw/visitors.csv")
rawclicks = pd.read_csv("../resources/wum_dataset_hw/clicks.csv")

In [171]:
# rawclicks[rawclicks["PageName"]=="HOWTOJOIN"]
# rawvisitors
# rawvisitors = rawvisitors[(rawvisitors["Length_pagecount"] >= 2)]
# rawvisitors
# rawclicks[rawclicks["VisitID"] ==  1226]
rawclicks[(rawclicks["PageName"] == "APPLICATION") & (rawclicks["SequenceNumber"] > 1)]
# rawvisitors[rawvisitors["VisitID"] ==  4643]


,VisitID,LocalID,PageID,PageName,CatName,CatID,ExtCatName,ExtCatID,TopicName,TopicID,TimeOnPage,PageScore,SequenceNumber
2224,2103,650684,3460,APPLICATION,Application,5,Application,9,In general,1000,30,62,3
4228,2914,652686,3678,APPLICATION,Application,5,Application,9,In general,1000,240,503,3
4740,3131,653200,3742,APPLICATION,Application,5,Application,9,In general,1000,30,71,4
12609,6612,661070,4250,APPLICATION,Application,5,Application,9,In general,1000,30,78,5
19332,9434,667790,4582,APPLICATION,Application,5,Application,9,In general,1000,60,125,3
20629,9936,669088,4627,APPLICATION,Application,5,Application,9,In general,1000,180,429,4
35073,15869,683533,5152,APPLICATION,Application,5,Application,9,In general,1000,30,62,3
36691,16484,685145,5200,APPLICATION,Application,5,Application,9,In general,1000,30,50,2
36692,16484,685151,5200,APPLICATION,Application,5,Application,9,In general,1000,240,739,8


In [172]:
rawclicks[rawclicks["VisitID"] ==  16484]

,VisitID,LocalID,PageID,PageName,CatName,CatID,ExtCatName,ExtCatID,TopicName,TopicID,TimeOnPage,PageScore,SequenceNumber
36684,16484,685144,3044,TravelAgency,Info,1,homepage,1,In general,1000,180,180,1
36685,16484,685146,3129,sightseeing tours,Search,2,Catalog,5,Tours with accommodation,8,60,125,3
36686,16484,685147,4204,England and London residence of English kings ...,Trip,3,undetected,-1,Tours with accommodation,8,60,143,4
36687,16484,685148,3169,Norway Norwegian fjords,Trip,3,undetected,-1,Tours with accommodation,8,60,156,5
36688,16484,685149,4204,England and London residence of English kings ...,Trip,3,undetected,-1,Tours with accommodation,8,30,83,6
36689,16484,685150,3135,Scotland United circuit Scotland,Trip,3,undetected,-1,Tours with accommodation,8,180,530,7
36690,16484,685152,3673,National Parks and the southern coast of Engla...,Trip,3,undetected,-1,Tourism,1,30,95,9
36691,16484,685145,5200,APPLICATION,Application,5,Application,9,In general,1000,30,50,2
36692,16484,685151,5200,APPLICATION,Application,5,Application,9,In general,1000,240,739,8


## Step 1

In [155]:
import pandas as pd

clicks = pd.read_csv("../resources/wum_dataset_hw/clicks.csv")

# drop page ids, local ids, cat ids and so on
clicks = clicks[
    [
        "VisitID",
        "SequenceNumber",
        "PageName",
        "CatName",
        "ExtCatName",
        "TopicName",
        "TimeOnPage",
    ]
]

def normalize(x):
    if pd.isna(x):
        return ""
    y = str(x).strip().upper()
    y = y.replace(" ", "_")
    return y

for col in ["PageName", "CatName", "ExtCatName", "TopicName"]:
    clicks[col] = clicks[col].apply(normalize)
# sort to preserve navigation order sequence, strip string and replace spaces with _
clicks = clicks.sort_values(["VisitID", "SequenceNumber"])

visitors = pd.read_csv("../resources/wum_dataset_hw/visitors.csv")
visitors = visitors[(visitors["Length_pagecount"] >= 2)]
valid_ids = set(visitors["VisitID"])

clicks = clicks[clicks["VisitID"].isin(valid_ids)]


In [156]:
PAGE_TO_APRIORI_CONVERSION = {
    "APPLICATION": "VIEW_APPLICATION", # macro
    "CATALOG": "VIEW_CATALOG",

    "INSURANCE": "VIEW_INSURANCE", # micro
    "DISCOUNT": "VIEW_DISCOUNT",
    "WHOWEARE": "VIEW_WHOWEARE",
    "HOWTOJOIN": "VIEW_HOWTOJOIN"
}

In [157]:
def process_session(session):
    items = set()

    topics = set()
    search_count = 0
    page_count = len(session)

    has_application = 0
    has_insurance = 0
    has_discount = 0
    has_catalog = 0
    has_whoweare = 0
    has_howtojoin = 0
    for page, cat, ext, topic, time in session:

        # ---- TOPICS, remove those which don't make sense or provide "INFO"----
        if topic not in {"IN_GENERAL", "UNDETECTED", "NENI", ""}:
            topics.add(topic)

        # ---- how many searches ----
        if cat == "SEARCH":
            search_count += 1

        # ---- CONVERSION / HELP PAGES ----
        mapped = None

        # using only pages as cat names don't include all (at least HOWTOJOIN)
        if page in PAGE_TO_APRIORI_CONVERSION:
            mapped = PAGE_TO_APRIORI_CONVERSION[page]

        if mapped:
            items.add(mapped)

            if mapped == "VIEW_APPLICATION":
                has_application = 1
            elif mapped == "VIEW_INSURANCE":
                has_insurance = 1
            elif mapped == "VIEW_DISCOUNT":
                has_discount = 1
            elif mapped == "VIEW_CATALOG":
                has_catalog = 1
            elif mapped == "VIEW_WHOWEARE":
                has_whoweare = 1
            elif mapped == "VIEW_HOWTOJOIN":
                has_howtojoin = 1

    # ---- ADD TOPICS TO TRANSACTION ----
    for t in topics:
        items.add(f"TOPIC_{t}")

    # ---- BEHAVIORAL FEATURES AS ITEMS ----
    for n in reversed(range(1, 9)):
        if len(topics) >= n:
            items.add(f"TOPICS>={n}")
            break
    for n in reversed(range(1, 9)):
        if search_count >= n:
            items.add(f"SEARCH>={n}")
            break

    # ---- FEATURE VECTOR ----
    features = {
        "Has_APPLICATION": has_application,
        "Has_INSURANCE": has_insurance,
        "Has_DISCOUNT": has_discount,
        "Has_CATALOG": has_catalog,
        "Has_WHOWEARE": has_whoweare,
        "Has_HOWTOJOIN": has_howtojoin,
        "SearchCount": search_count,
        "PageCount": page_count,
        "TopicCount": len(topics),
    }

    return items, features

In [158]:
def build_sessions(df):
    return list(zip(
        df["PageName"],
        df["CatName"],
        df["ExtCatName"],
        df["TopicName"],
        df["TimeOnPage"]
    ))

sessions = clicks.groupby("VisitID").apply(build_sessions, include_groups=False)
sessions.reset_index(name='actions')

,VisitID,actions
0,1185,"[(TRAVELAGENCY, INFO, HOMEPAGE, IN_GENERAL, 30..."
1,1186,"[(NORWAY_JOURNEY_TO_THE_FAR_NORTH_LOFOTEN, TRI..."
2,1187,"[(TRAVELAGENCY, INFO, HOMEPAGE, IN_GENERAL, 30..."
3,1189,"[(TRAVELAGENCY, INFO, HOMEPAGE, IN_GENERAL, 60..."
4,1190,"[(N_NEWS_TRIPS, INFO, SENDS_NEWS, IN_GENERAL, ..."
...,...,...
6639,17257,"[(TRAVELAGENCY, INFO, HOMEPAGE, IN_GENERAL, 60..."
6640,17258,"[(TRAVELAGENCY, INFO, HOMEPAGE, IN_GENERAL, 30..."
6641,17260,"[(TRAVELAGENCY, INFO, HOMEPAGE, IN_GENERAL, 30..."
6642,17264,"[(BULGARIA_STAY_WITH_HOLIDAY_TRIPS_05, TRIP, U..."


In [164]:
transaction_list = []
feature_rows = []

for visit_id, session in sessions.items():
    items, features = process_session(session)
    transaction_list.append(
        {"VisitId": visit_id, "items": items}
    )

    features["VisitID"] = visit_id
    feature_rows.append(features)

transactions_df = pd.DataFrame(transaction_list)
features_df = pd.DataFrame(feature_rows)

In [174]:
# transactions_df[(features_df['Has_APPLICATION' ] == True) | (features_df['Has_CATALOG'] == True)]
transactions_df[(features_df['Has_APPLICATION'] == True)]


,VisitId,items
32,1259,"{TOPIC_TOURISM, TOPICS>=2, VIEW_CATALOG, SEARC..."
398,2103,"{VIEW_APPLICATION, TOPIC_EXOTICA, TOPICS>=1, S..."
757,2914,"{TOPICS>=1, VIEW_DISCOUNT, VIEW_APPLICATION, T..."
838,3131,"{SEARCH>=1, TOPICS>=2, TOPIC_SKI, TOPIC_LASTMI..."
1163,4006,"{TOPIC_TOURISM, VIEW_APPLICATION, TOPICS>=1, S..."
2143,6612,"{VIEW_WHOWEARE, VIEW_APPLICATION, SEARCH>=1}"
2857,8505,"{TOPIC_SKI, VIEW_APPLICATION, TOPICS>=1}"
3250,9434,"{TOPIC_HIKING, TOPICS>=2, TOPIC_ALPS, SEARCH>=..."
3457,9936,"{TOPIC_TOURS_WITH_ACCOMMODATION, SEARCH>=1, TO..."
3826,10842,"{TOPIC_TOURS_WITH_ACCOMMODATION, TOPICS>=3, TO..."


In [163]:
from collections import Counter

item_counts = Counter()
for t in transaction_list:
    item_counts.update(t["items"])

print(item_counts.most_common(50))

[('TOPICS>=1', 2876), ('SEARCH>=1', 2580), ('TOPICS>=2', 1758), ('SEARCH>=2', 1556), ('TOPIC_TOURISM', 1397), ('TOPIC_LASTMINUTE', 1245), ('TOPIC_TOURS_WITH_ACCOMMODATION', 1051), ('TOPIC_HIKING', 1017), ('TOPIC_EXOTICA', 873), ('TOPICS>=3', 758), ('SEARCH>=3', 750), ('TOPIC_CORSICA', 604), ('TOPIC_CYCLO', 555), ('TOPIC_CHALLENGING_EXPEDITION', 517), ('TOPIC_TOURS_WITH_LIGHTWEIGHT_TOURISM', 489), ('TOPIC_TOURIST', 469), ('TOPIC_VHT', 467), ('TOPIC_BULGARIA', 438), ('TOPIC_STAYS_WITH_TRIPS', 435), ('TOPIC_SKI', 423), ('TOPIC_LIPARI', 406), ('VIEW_CATALOG', 394), ('SEARCH>=4', 366), ('TOPICS>=4', 309), ('TOPIC_HOTELBUSES', 284), ('TOPIC_MONTENEGRO', 284), ('TOPIC_ALPS', 248), ('TOPIC_HOLIDAY_SHIPPING', 226), ('TOPIC_WATER', 208), ('TOPIC_RAFTING', 205), ('SEARCH>=5', 199), ('TOPICS>=5', 180), ('TOPIC_CLIMBING_SCHOOL', 174), ('VIEW_DISCOUNT', 172), ('VIEW_WHOWEARE', 159), ('SEARCH>=8', 125), ('SEARCH>=6', 100), ('TOPICS>=6', 79), ('TOPIC_SCHOOL_TOURS', 79), ('TOPIC_GOLF', 67), ('SEARCH>=7